# Fine-tune DistilBERT for pandas GitHub Issues

This notebook trains a 5-class issue classifier on `pandas-dev/pandas`, shows a few quick data visualizations, and saves the final transformer artifacts into `model_server/models/fine_tuned/distilbert_pandas_issues/`.

The notebook prefers the local cached dataset at `data_pipeline/pandas_issues.json`. If that file is missing, it falls back to the GitHub API so the workflow still works in a fresh clone.

In [ ]:
!pip install -q datasets transformers scikit-learn pandas requests tqdm matplotlib seaborn huggingface_hub accelerate torch tokenizers


In [ ]:
from __future__ import annotations

import json
import os
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm
from tokenizers import BertWordPieceTokenizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BertTokenizerFast, DistilBertConfig, DistilBertForSequenceClassification, Trainer, TrainingArguments

sns.set_theme(style='whitegrid', palette='deep')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'data_pipeline').exists() else NOTEBOOK_DIR.parent
CACHE_PATH = REPO_ROOT / 'data_pipeline' / 'pandas_issues.json'
OUTPUT_DIR = REPO_ROOT / 'model_server' / 'models' / 'fine_tuned' / 'distilbert_pandas_issues'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'distilbert-base-uncased'
LABELS = ['bug', 'feature', 'docs', 'question', 'other']
label_to_id = {label: idx for idx, label in enumerate(LABELS)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
LOCAL_TOKENIZER_DIR = OUTPUT_DIR / 'local_tokenizer'
LOCAL_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)


def build_local_tokenizer(texts: Iterable[str]) -> BertTokenizerFast:
    corpus_path = LOCAL_TOKENIZER_DIR / 'corpus.txt'
    corpus_path.write_text('\n'.join(texts), encoding='utf-8')
    trainer = BertWordPieceTokenizer(lowercase=True)
    trainer.train(
        files=[str(corpus_path)],
        vocab_size=8000,
        min_frequency=1,
        special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    )
    trainer.save_model(str(LOCAL_TOKENIZER_DIR))
    return BertTokenizerFast.from_pretrained(str(LOCAL_TOKENIZER_DIR), do_lower_case=True)


def load_tokenizer(frame: pd.DataFrame) -> tuple[BertTokenizerFast | AutoTokenizer, str]:
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
        return tokenizer, f'pretrained:{MODEL_NAME}'
    except Exception as exc:
        print(f'Falling back to a local WordPiece tokenizer: {exc}')
        tokenizer = build_local_tokenizer(frame['text'].fillna('').astype(str).tolist())
        return tokenizer, 'local_wordpiece'


def load_model(tokenizer) -> tuple[DistilBertForSequenceClassification | AutoModelForSequenceClassification, str]:
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=len(LABELS),
            id2label=id_to_label,
            label2id=label_to_id,
            local_files_only=True,
        )
        return model, f'pretrained:{MODEL_NAME}'
    except Exception as exc:
        print(f'Falling back to a locally initialized DistilBERT model: {exc}')
        config = DistilBertConfig(
            vocab_size=len(tokenizer),
            dim=192,
            hidden_dim=384,
            n_layers=2,
            n_heads=4,
            dropout=0.1,
            attention_dropout=0.1,
            num_labels=len(LABELS),
            id2label=id_to_label,
            label2id=label_to_id,
        )
        model = DistilBertForSequenceClassification(config)
        return model, 'local_distilbert_scratch'

print('Repo root:', REPO_ROOT)
print('Artifacts:', OUTPUT_DIR)


In [ ]:
def fetch_pandas_issues(limit: int = 2000) -> list[dict[str, object]]:
    headers = {'Accept': 'application/vnd.github+json'}
    token = os.getenv('GITHUB_TOKEN')
    if token:
        headers['Authorization'] = f'Bearer {token}'

    issues: list[dict[str, object]] = []
    page = 1
    per_page = 100
    with tqdm(total=limit, desc='Fetching issues') as pbar:
        while len(issues) < limit and page <= 30:
            response = requests.get(
                'https://api.github.com/repos/pandas-dev/pandas/issues',
                params={'state': 'closed', 'per_page': per_page, 'page': page},
                headers=headers,
                timeout=30,
            )
            response.raise_for_status()
            batch = response.json()
            if not batch:
                break

            for item in batch:
                if 'pull_request' in item:
                    continue
                issues.append(
                    {
                        'id': item['id'],
                        'title': item['title'],
                        'body': item.get('body') or '',
                        'labels': [label['name'] for label in item.get('labels', [])],
                        'created_at': item['created_at'],
                        'state': item['state'],
                    }
                )
                pbar.update(1)
                if len(issues) >= limit:
                    break

            page += 1

    return issues


def load_issues() -> list[dict[str, object]]:
    if CACHE_PATH.exists():
        return json.loads(CACHE_PATH.read_text(encoding='utf-8'))

    issues = fetch_pandas_issues()
    CACHE_PATH.write_text(json.dumps(issues, indent=2), encoding='utf-8')
    return issues


issues = load_issues()
print(f'Loaded {len(issues)} issues')


In [ ]:
def map_label(labels: list[object]) -> str:
    lowered = [str(label).lower() for label in labels]
    if any(label in lowered for label in ['bug', 'regression']):
        return 'bug'
    if any(label in lowered for label in ['enhancement', 'feature', 'new feature']):
        return 'feature'
    if any(label in lowered for label in ['docs', 'documentation']):
        return 'docs'
    if any(label in lowered for label in ['question', 'how-to']):
        return 'question'
    return 'other'


rows = []
for issue in issues:
    label = map_label(issue.get('labels') or [])
    text = f"{issue.get('title', '')}\n{issue.get('body', '') or ''}".strip()
    rows.append(
        {
            'id': issue.get('id'),
            'title': issue.get('title', ''),
            'body': issue.get('body', ''),
            'created_at': issue.get('created_at'),
            'label': label,
            'text': text,
        }
    )

df = pd.DataFrame(rows)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', utc=True)
df['text_length'] = df['text'].str.len()
df['label_id'] = df['label'].map(label_to_id)
df = df.dropna(subset=['label_id']).copy()
df['label_id'] = df['label_id'].astype(int)

print(df['label'].value_counts().reindex(LABELS, fill_value=0))
df.head(3)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='label', order=LABELS, ax=axes[0])
axes[0].set_title('Label distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

sns.histplot(data=df, x='text_length', bins=40, ax=axes[1], color='#4C72B0')
axes[1].set_title('Issue text length')
axes[1].set_xlabel('Characters')

sns.boxplot(data=df, x='label', y='text_length', order=LABELS, ax=axes[2])
axes[2].set_title('Length by class')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Characters')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=180, bbox_inches='tight')
plt.show()

df.sample(min(5, len(df)))[['created_at', 'label', 'title']]


In [ ]:
def parse_created_at(value: object) -> datetime:
    raw = str(value or '')
    try:
        return datetime.fromisoformat(raw.replace('Z', '+00:00'))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def time_stratified_split(frame: pd.DataFrame, train_ratio: float = 0.6, val_ratio: float = 0.2):
    grouped: dict[str, list[pd.Series]] = defaultdict(list)
    for _, row in frame.iterrows():
        grouped[str(row['label'])].append(row)

    splits = {'train': [], 'val': [], 'test': []}
    for rows in grouped.values():
        rows = sorted(rows, key=lambda row: parse_created_at(row['created_at']))
        total = len(rows)
        if total == 1:
            splits['train'].extend(rows)
            continue

        train_count = max(1, int(round(total * train_ratio)))
        val_count = max(1 if total >= 3 else 0, int(round(total * val_ratio)))
        if train_count + val_count >= total:
            overflow = train_count + val_count - (total - 1)
            if overflow > 0:
                reduction = min(overflow, max(train_count - 1, 0))
                train_count -= reduction
                overflow -= reduction
            if overflow > 0:
                val_count -= min(overflow, max(val_count - 1, 0))

        test_count = total - train_count - val_count
        if test_count <= 0 and total >= 3:
            if val_count > 1:
                val_count -= 1
            elif train_count > 1:
                train_count -= 1

        splits['train'].extend(rows[:train_count])
        splits['val'].extend(rows[train_count:train_count + val_count])
        splits['test'].extend(rows[train_count + val_count:])

    return (
        pd.DataFrame(splits['train']).reset_index(drop=True),
        pd.DataFrame(splits['val']).reset_index(drop=True),
        pd.DataFrame(splits['test']).reset_index(drop=True),
    )


train_df, val_df, test_df = time_stratified_split(df)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(train_df['label'].value_counts().reindex(LABELS, fill_value=0))


In [ ]:
tokenizer, tokenizer_source = load_tokenizer(train_df)


def as_dataset(frame: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(frame[['text', 'label_id']], preserve_index=False)


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)


datasets = DatasetDict(
    {
        'train': as_dataset(train_df),
        'validation': as_dataset(val_df),
        'test': as_dataset(test_df),
    }
)
datasets = datasets.map(tokenize, batched=True, remove_columns=['text'])
datasets = datasets.rename_column('label_id', 'labels')
datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print(datasets)


In [ ]:
model, model_source = load_model(tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['validation'],
    compute_metrics=compute_metrics,
)

trainer


In [ ]:
train_result = trainer.train()
train_result.metrics


In [ ]:
test_metrics = trainer.evaluate(datasets['test'])
test_metrics


In [ ]:
predictions = trainer.predict(datasets['test'])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print(classification_report(y_true, y_pred, labels=list(range(len(LABELS))), target_names=LABELS, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title('DistilBERT confusion matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

summary = {
    'model_name': MODEL_NAME,
    'tokenizer_source': tokenizer_source,
    'model_source': model_source,
    'labels': LABELS,
    'splits': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
    'test_metrics': {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float, np.floating))},
}
metrics_path = OUTPUT_DIR / 'metrics.json'
label_map_path = OUTPUT_DIR / 'label_map.json'
summary_path = OUTPUT_DIR / 'training_summary.json'

metrics_path.write_text(json.dumps(summary['test_metrics'], indent=2), encoding='utf-8')
label_map_path.write_text(json.dumps(label_to_id, indent=2), encoding='utf-8')
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

summary


## What to do next

- Keep the saved directory in `model_server/models/fine_tuned/distilbert_pandas_issues/`.
- If you want to expose the transformer model through the API, I can wire a `model=dl` classification path next.
- The current notebook already gives you the EDA plots, the training run, and the saved transformer artifacts.